<a href="https://colab.research.google.com/github/dayalupihu/FastSpeech2-English-Telugu/blob/main/FastSpeech2_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 print("Hello! Welcome to FastSpeech 2 Assignment.")

name = "FastSpeech 2"

print("Today we are learning:", name)

Hello! Welcome to FastSpeech 2 Assignment.
Today we are learning: FastSpeech 2


In [ ]:
import torch

print("PyTorch version:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("No GPU detected. Check your Colab runtime settings.")


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!git clone https://github.com/ming024/FastSpeech2.git

Cloning into 'FastSpeech2'...
remote: Enumerating objects: 994, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 994 (delta 11), reused 7 (delta 7), pack-reused 978 (from 1)
Receiving objects: 100% (994/994), 330.31 MiB | 17.77 MiB/s, done.
Resolving deltas: 100% (176/176), done.


In [ ]:
%cd FastSpeech2

/content/FastSpeech2


In [ ]:
!ls

audio	     hifigan	 model		    README.md	      transformer
config	     img	 prepare_align.py   requirements.txt  utils
dataset.py   index.html  preprocessed_data  synthesize.py
demo	     lexicon	 preprocessor	    text
evaluate.py  LICENSE	 preprocess.py	    train.py


In [ ]:
!python --version

Python 3.13.16


In [ ]:
!cat requirements.txt

g2p-en == 2.1.0
inflect == 4.1.0
librosa == 0.7.2
matplotlib == 3.2.2
numba == 0.48
numpy == 1.19.0
pypinyin==0.39.0
pyworld == 0.2.10
PyYAML==5.4.1
scikit-learn==0.23.2
scipy == 1.5.0
soundfile==0.10.3.post1
tensorboard == 2.2.2
tgt == 1.4.4
torch == 1.7.0
tqdm==4.46.1
unidecode == 1.1.1

In [ ]:
!find config -maxdepth 2 -type f

config/LJSpeech/model.yaml
config/LJSpeech/preprocess.yaml
config/LJSpeech/train.yaml
config/LJSpeech_paper/model.yaml
config/LJSpeech_paper/preprocess.yaml
config/LJSpeech_paper/train.yaml
config/AISHELL3/model.yaml
config/AISHELL3/preprocess.yaml
config/AISHELL3/train.yaml
config/README.md
config/LibriTTS/model.yaml
config/LibriTTS/preprocess.yaml
config/LibriTTS/train.yaml


In [ ]:
!ls model

fastspeech2.py	__init__.py  loss.py  modules.py  optimizer.py


In [ ]:
!grep -n "^class " model/fastspeech2.py model/modules.py

model/fastspeech2.py:13:class FastSpeech2(nn.Module):
model/modules.py:17:class VarianceAdaptor(nn.Module):
model/modules.py:161:class LengthRegulator(nn.Module):
model/modules.py:197:class VariancePredictor(nn.Module):
model/modules.py:253:class Conv(nn.Module):


In [ ]:
!cat config/LJSpeech/model.yaml

transformer:
  encoder_layer: 4
  encoder_head: 2
  encoder_hidden: 256
  decoder_layer: 6
  decoder_head: 2
  decoder_hidden: 256
  conv_filter_size: 1024
  conv_kernel_size: [9, 1]
  encoder_dropout: 0.2
  decoder_dropout: 0.2

variance_predictor:
  filter_size: 256
  kernel_size: 3
  dropout: 0.5

variance_embedding:
  pitch_quantization: "linear" # support 'linear' or 'log', 'log' is allowed only if the pitch values are not normalized during preprocessing
  energy_quantization: "linear" # support 'linear' or 'log', 'log' is allowed only if the energy values are not normalized during preprocessing
  n_bins: 256

# gst:
#   use_gst: False
#   conv_filters: [32, 32, 64, 64, 128, 128]
#   gru_hidden: 128
#   token_size: 128
#   n_style_token: 10
#   attn_head: 4

multi_speaker: False

max_seq_len: 1000

vocoder:
  model: "HiFi-GAN" # support 'HiFi-GAN', 'MelGAN'
  speaker: "LJSpeech" # support  'LJSpeech', 'universal'


In [ ]:
!cat config/LJSpeech/preprocess.yaml

dataset: "LJSpeech"

path:
  corpus_path: "/home/ming/Data/LJSpeech-1.1"
  lexicon_path: "lexicon/librispeech-lexicon.txt"
  raw_path: "./raw_data/LJSpeech"
  preprocessed_path: "./preprocessed_data/LJSpeech"

preprocessing:
  val_size: 512
  text:
    text_cleaners: ["english_cleaners"]
    language: "en"
  audio:
    sampling_rate: 22050
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: 80
    mel_fmin: 0
    mel_fmax: 8000 # please set to 8000 for HiFi-GAN vocoder, set to null for MelGAN vocoder
  pitch:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True
  energy:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True


In [ ]:
import torch
import numpy
import librosa

print("Torch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("Librosa:", librosa.__version__)

Torch: 2.14.1+cu130
NumPy: 2.4.6
Librosa: 0.11.0


In [ ]:
!sed -n '1,130p' model/fastspeech2.py

import os
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformer import Encoder, Decoder, PostNet
from .modules import VarianceAdaptor
from utils.tools import get_mask_from_lengths


class FastSpeech2(nn.Module):
    """ FastSpeech2 """

    def __init__(self, preprocess_config, model_config):
        super(FastSpeech2, self).__init__()
        self.model_config = model_config

        self.encoder = Encoder(model_config)
        self.variance_adaptor = VarianceAdaptor(preprocess_config, model_config)
        self.decoder = Decoder(model_config)
        self.mel_linear = nn.Linear(
            model_config["transformer"]["decoder_hidden"],
            preprocess_config["preprocessing"]["mel"]["n_mel_channels"],
        )
        self.postnet = PostNet()

        self.speaker_emb = None
        if model_config["multi_speaker"]:
            with open(
                os.path.join(
                    preprocess_config["path"]["preprocessed_path"

In [ ]:
!sed -n '17,160p' model/modules.py

class VarianceAdaptor(nn.Module):
    """Variance Adaptor"""

    def __init__(self, preprocess_config, model_config):
        super(VarianceAdaptor, self).__init__()
        self.duration_predictor = VariancePredictor(model_config)
        self.length_regulator = LengthRegulator()
        self.pitch_predictor = VariancePredictor(model_config)
        self.energy_predictor = VariancePredictor(model_config)

        self.pitch_feature_level = preprocess_config["preprocessing"]["pitch"][
            "feature"
        ]
        self.energy_feature_level = preprocess_config["preprocessing"]["energy"][
            "feature"
        ]
        assert self.pitch_feature_level in ["phoneme_level", "frame_level"]
        assert self.energy_feature_level in ["phoneme_level", "frame_level"]

        pitch_quantization = model_config["variance_embedding"]["pitch_quantization"]
        energy_quantization = model_config["variance_embedding"]["energy_quantization"]
        n_bins = model_config["vari

In [ ]:
!sed -n '161,196p' model/modules.py

class LengthRegulator(nn.Module):
    """Length Regulator"""

    def __init__(self):
        super(LengthRegulator, self).__init__()

    def LR(self, x, duration, max_len):
        output = list()
        mel_len = list()
        for batch, expand_target in zip(x, duration):
            expanded = self.expand(batch, expand_target)
            output.append(expanded)
            mel_len.append(expanded.shape[0])

        if max_len is not None:
            output = pad(output, max_len)
        else:
            output = pad(output)

        return output, torch.LongTensor(mel_len).to(device)

    def expand(self, batch, predicted):
        out = list()

        for i, vec in enumerate(batch):
            expand_size = predicted[i].item()
            out.append(vec.expand(max(int(expand_size), 0), -1))
        out = torch.cat(out, 0)

        return out

    def forward(self, x, duration, max_len):
        output, mel_len = self.LR(x, duration, max_len)
        return output, mel_len



In [ ]:
!sed -n '197,252p' model/modules.py

class VariancePredictor(nn.Module):
    """Duration, Pitch and Energy Predictor"""

    def __init__(self, model_config):
        super(VariancePredictor, self).__init__()

        self.input_size = model_config["transformer"]["encoder_hidden"]
        self.filter_size = model_config["variance_predictor"]["filter_size"]
        self.kernel = model_config["variance_predictor"]["kernel_size"]
        self.conv_output_size = model_config["variance_predictor"]["filter_size"]
        self.dropout = model_config["variance_predictor"]["dropout"]

        self.conv_layer = nn.Sequential(
            OrderedDict(
                [
                    (
                        "conv1d_1",
                        Conv(
                            self.input_size,
                            self.filter_size,
                            kernel_size=self.kernel,
                            padding=(self.kernel - 1) // 2,
                        ),
                    ),
                    ("relu_1

In [ ]:
output = self.encoder(texts, src_masks)

NameError: name 'self' is not defined

In [ ]:
!sed -n '1,200p' dataset.py

import json
import math
import os

import numpy as np
from torch.utils.data import Dataset

from text import text_to_sequence
from utils.tools import pad_1D, pad_2D


class Dataset(Dataset):
    def __init__(
        self, filename, preprocess_config, train_config, sort=False, drop_last=False
    ):
        self.dataset_name = preprocess_config["dataset"]
        self.preprocessed_path = preprocess_config["path"]["preprocessed_path"]
        self.cleaners = preprocess_config["preprocessing"]["text"]["text_cleaners"]
        self.batch_size = train_config["optimizer"]["batch_size"]

        self.basename, self.speaker, self.text, self.raw_text = self.process_meta(
            filename
        )
        with open(os.path.join(self.preprocessed_path, "speakers.json")) as f:
            self.speaker_map = json.load(f)
        self.sort = sort
        self.drop_last = drop_last

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        basename = self.bas

In [ ]:
!sed -n '1,180p' preprocess.py

import argparse

import yaml

from preprocessor.preprocessor import Preprocessor


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("config", type=str, help="path to preprocess.yaml")
    args = parser.parse_args()

    config = yaml.load(open(args.config, "r"), Loader=yaml.FullLoader)
    preprocessor = Preprocessor(config)
    preprocessor.build_from_path()


In [ ]:
!ls -R preprocessor

ls: cannot access 'preprocessor': No such file or directory


In [ ]:
!sed -n '1,240p' preprocessor/preprocessor.py

sed: can't read preprocessor/preprocessor.py: No such file or directory


In [ ]:
!find preprocessor -maxdepth 2 -type f

find: ‘preprocessor’: No such file or directory


In [ ]:
!ls -R preprocessor

ls: cannot access 'preprocessor': No such file or directory


In [ ]:
!pwd
!ls

/content
sample_data


In [ ]:
%cd /content/FastSpeech2

[Errno 2] No such file or directory: '/content/FastSpeech2'
/content


In [ ]:
%cd /content

/content


In [ ]:
%cd /content/FastSpeech2

[Errno 2] No such file or directory: '/content/FastSpeech2'
/content


In [ ]:
!ls -lah /content

total 16K
drwxr-xr-x 1 root root 4.0K Nov  6 16:54 .
drwxr-xr-x 1 root root 4.0K Nov  7 10:50 ..
drwxr-xr-x 4 root root 4.0K Nov  6 16:54 .config
drwxr-xr-x 1 root root 4.0K Nov  6 16:54 sample_data


In [ ]:
!find /content -maxdepth 3 -type d -name "FastSpeech2"

In [ ]:
!pwd
!ls -lah /content

/content
total 16K
drwxr-xr-x 1 root root 4.0K Nov  6 16:54 .
drwxr-xr-x 1 root root 4.0K Nov  7 10:50 ..
drwxr-xr-x 4 root root 4.0K Nov  6 16:54 .config
drwxr-xr-x 1 root root 4.0K Nov  6 16:54 sample_data


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
!git clone https://github.com/ming024/FastSpeech2.git
%cd /content/FastSpeech2
!ls

Cloning into 'FastSpeech2'...
remote: Enumerating objects: 994, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 994 (delta 11), reused 7 (delta 7), pack-reused 978 (from 1)
Receiving objects: 100% (994/994), 330.31 MiB | 23.96 MiB/s, done.
Resolving deltas: 100% (176/176), done.
/content/FastSpeech2
audio	     hifigan	 model		    README.md	      transformer
config	     img	 prepare_align.py   requirements.txt  utils
dataset.py   index.html  preprocessed_data  synthesize.py
demo	     lexicon	 preprocessor	    text
evaluate.py  LICENSE	 preprocess.py	    train.py


In [ ]:
!ls

audio	     hifigan	 model		    README.md	      transformer
config	     img	 prepare_align.py   requirements.txt  utils
dataset.py   index.html  preprocessed_data  synthesize.py
demo	     lexicon	 preprocessor	    text
evaluate.py  LICENSE	 preprocess.py	    train.py


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4


In [ ]:
%cd /content/FastSpeech2

!pwd
!ls

/content/FastSpeech2
/content/FastSpeech2
audio	     hifigan	 model		    README.md	      transformer
config	     img	 prepare_align.py   requirements.txt  utils
dataset.py   index.html  preprocessed_data  synthesize.py
demo	     lexicon	 preprocessor	    text
evaluate.py  LICENSE	 preprocess.py	    train.py


In [ ]:
!ls -la preprocessor

total 32
drwxr-xr-x  2 root root  4096 Nov  7 11:01 .
drwxr-xr-x 15 root root  4096 Nov  7 11:01 ..
-rw-r--r--  1 root root  1480 Nov  7 11:01 aishell3.py
-rw-r--r--  1 root root  1768 Nov  7 11:01 libritts.py
-rw-r--r--  1 root root  1440 Nov  7 11:01 ljspeech.py
-rw-r--r--  1 root root 10912 Nov  7 11:01 preprocessor.py


In [ ]:
%cd /content/FastSpeech2

!pwd
!ls

/content/FastSpeech2
/content/FastSpeech2
audio	     hifigan	 model		    README.md	      transformer
config	     img	 prepare_align.py   requirements.txt  utils
dataset.py   index.html  preprocessed_data  synthesize.py
demo	     lexicon	 preprocessor	    text
evaluate.py  LICENSE	 preprocess.py	    train.py


In [ ]:
!ls -la preprocessor

total 32
drwxr-xr-x  2 root root  4096 Nov  7 11:01 .
drwxr-xr-x 15 root root  4096 Nov  7 11:01 ..
-rw-r--r--  1 root root  1480 Nov  7 11:01 aishell3.py
-rw-r--r--  1 root root  1768 Nov  7 11:01 libritts.py
-rw-r--r--  1 root root  1440 Nov  7 11:01 ljspeech.py
-rw-r--r--  1 root root 10912 Nov  7 11:01 preprocessor.py


In [ ]:
!sed -n '1,240p' preprocessor/preprocessor.py

import os
import random
import json

import tgt
import librosa
import numpy as np
import pyworld as pw
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import audio as Audio


class Preprocessor:
    def __init__(self, config):
        self.config = config
        self.in_dir = config["path"]["raw_path"]
        self.out_dir = config["path"]["preprocessed_path"]
        self.val_size = config["preprocessing"]["val_size"]
        self.sampling_rate = config["preprocessing"]["audio"]["sampling_rate"]
        self.hop_length = config["preprocessing"]["stft"]["hop_length"]

        assert config["preprocessing"]["pitch"]["feature"] in [
            "phoneme_level",
            "frame_level",
        ]
        assert config["preprocessing"]["energy"]["feature"] in [
            "phoneme_level",
            "frame_level",
        ]
        self.pitch_phoneme_averaging = (
            config["preprocessing"]["pitch"]["feature"] == 

In [ ]:
!cat preprocessor/ljspeech.py

import os

import librosa
import numpy as np
from scipy.io import wavfile
from tqdm import tqdm

from text import _clean_text


def prepare_align(config):
    in_dir = config["path"]["corpus_path"]
    out_dir = config["path"]["raw_path"]
    sampling_rate = config["preprocessing"]["audio"]["sampling_rate"]
    max_wav_value = config["preprocessing"]["audio"]["max_wav_value"]
    cleaners = config["preprocessing"]["text"]["text_cleaners"]
    speaker = "LJSpeech"
    with open(os.path.join(in_dir, "metadata.csv"), encoding="utf-8") as f:
        for line in tqdm(f):
            parts = line.strip().split("|")
            base_name = parts[0]
            text = parts[2]
            text = _clean_text(text, cleaners)

            wav_path = os.path.join(in_dir, "wavs", "{}.wav".format(base_name))
            if os.path.exists(wav_path):
                os.makedirs(os.path.join(out_dir, speaker), exist_ok=True)
                wav, _ = librosa.load(wav_path, sampling_rate)
               

In [ ]:
!cat config/LJSpeech/preprocess.yaml

dataset: "LJSpeech"

path:
  corpus_path: "/home/ming/Data/LJSpeech-1.1"
  lexicon_path: "lexicon/librispeech-lexicon.txt"
  raw_path: "./raw_data/LJSpeech"
  preprocessed_path: "./preprocessed_data/LJSpeech"

preprocessing:
  val_size: 512
  text:
    text_cleaners: ["english_cleaners"]
    language: "en"
  audio:
    sampling_rate: 22050
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: 80
    mel_fmin: 0
    mel_fmax: 8000 # please set to 8000 for HiFi-GAN vocoder, set to null for MelGAN vocoder
  pitch:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True
  energy:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True


In [ ]:
!sed -n '1,240p' preprocessor/preprocessor.py

import os
import random
import json

import tgt
import librosa
import numpy as np
import pyworld as pw
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import audio as Audio


class Preprocessor:
    def __init__(self, config):
        self.config = config
        self.in_dir = config["path"]["raw_path"]
        self.out_dir = config["path"]["preprocessed_path"]
        self.val_size = config["preprocessing"]["val_size"]
        self.sampling_rate = config["preprocessing"]["audio"]["sampling_rate"]
        self.hop_length = config["preprocessing"]["stft"]["hop_length"]

        assert config["preprocessing"]["pitch"]["feature"] in [
            "phoneme_level",
            "frame_level",
        ]
        assert config["preprocessing"]["energy"]["feature"] in [
            "phoneme_level",
            "frame_level",
        ]
        self.pitch_phoneme_averaging = (
            config["preprocessing"]["pitch"]["feature"] == 

In [ ]:
!cat preprocessor/ljspeech.py

import os

import librosa
import numpy as np
from scipy.io import wavfile
from tqdm import tqdm

from text import _clean_text


def prepare_align(config):
    in_dir = config["path"]["corpus_path"]
    out_dir = config["path"]["raw_path"]
    sampling_rate = config["preprocessing"]["audio"]["sampling_rate"]
    max_wav_value = config["preprocessing"]["audio"]["max_wav_value"]
    cleaners = config["preprocessing"]["text"]["text_cleaners"]
    speaker = "LJSpeech"
    with open(os.path.join(in_dir, "metadata.csv"), encoding="utf-8") as f:
        for line in tqdm(f):
            parts = line.strip().split("|")
            base_name = parts[0]
            text = parts[2]
            text = _clean_text(text, cleaners)

            wav_path = os.path.join(in_dir, "wavs", "{}.wav".format(base_name))
            if os.path.exists(wav_path):
                os.makedirs(os.path.join(out_dir, speaker), exist_ok=True)
                wav, _ = librosa.load(wav_path, sampling_rate)
               

In [ ]:
!cat preprocessor/ljspeech.py

import os

import librosa
import numpy as np
from scipy.io import wavfile
from tqdm import tqdm

from text import _clean_text


def prepare_align(config):
    in_dir = config["path"]["corpus_path"]
    out_dir = config["path"]["raw_path"]
    sampling_rate = config["preprocessing"]["audio"]["sampling_rate"]
    max_wav_value = config["preprocessing"]["audio"]["max_wav_value"]
    cleaners = config["preprocessing"]["text"]["text_cleaners"]
    speaker = "LJSpeech"
    with open(os.path.join(in_dir, "metadata.csv"), encoding="utf-8") as f:
        for line in tqdm(f):
            parts = line.strip().split("|")
            base_name = parts[0]
            text = parts[2]
            text = _clean_text(text, cleaners)

            wav_path = os.path.join(in_dir, "wavs", "{}.wav".format(base_name))
            if os.path.exists(wav_path):
                os.makedirs(os.path.join(out_dir, speaker), exist_ok=True)
                wav, _ = librosa.load(wav_path, sampling_rate)
               

In [ ]:
!sed -n '241,360p' preprocessor/preprocessor.py

        np.save(
            os.path.join(self.out_dir, "mel", mel_filename),
            mel_spectrogram.T,
        )

        return (
            "|".join([basename, speaker, text, raw_text]),
            self.remove_outlier(pitch),
            self.remove_outlier(energy),
            mel_spectrogram.shape[1],
        )

    def get_alignment(self, tier):
        sil_phones = ["sil", "sp", "spn"]

        phones = []
        durations = []
        start_time = 0
        end_time = 0
        end_idx = 0
        for t in tier._objects:
            s, e, p = t.start_time, t.end_time, t.text

            # Trim leading silences
            if phones == []:
                if p in sil_phones:
                    continue
                else:
                    start_time = s

            if p not in sil_phones:
                # For ordinary phones
                phones.append(p)
                end_time = e
                end_idx = len(phones)
            else:
                # For

In [ ]:
!cat preprocess.py

import argparse

import yaml

from preprocessor.preprocessor import Preprocessor


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("config", type=str, help="path to preprocess.yaml")
    args = parser.parse_args()

    config = yaml.load(open(args.config, "r"), Loader=yaml.FullLoader)
    preprocessor = Preprocessor(config)
    preprocessor.build_from_path()


In [ ]:
!cat text/__init__.py

""" from https://github.com/keithito/tacotron """
import re
from text import cleaners
from text.symbols import symbols


# Mappings from symbol to numeric ID and vice versa:
_symbol_to_id = {s: i for i, s in enumerate(symbols)}
_id_to_symbol = {i: s for i, s in enumerate(symbols)}

# Regular expression matching text enclosed in curly braces:
_curly_re = re.compile(r"(.*?)\{(.+?)\}(.*)")


def text_to_sequence(text, cleaner_names):
    """Converts a string of text to a sequence of IDs corresponding to the symbols in the text.

    The text can optionally have ARPAbet sequences enclosed in curly braces embedded
    in it. For example, "Turn left on {HH AW1 S S T AH0 N} Street."

    Args:
      text: string to convert to a sequence
      cleaner_names: names of the cleaner functions to run the text through

    Returns:
      List of integers corresponding to the symbols in the text
    """
    sequence = []

    # Check for curly braces and treat their contents as ARPAbet:
    while len

In [ ]:
!cat text/symbols.py

""" from https://github.com/keithito/tacotron """

"""
Defines the set of symbols used in text input to the model.

The default is a set of ASCII characters that works well for English or text that has been run through Unidecode. For other data, you can modify _characters. See TRAINING_DATA.md for details. """

from text import cmudict, pinyin

_pad = "_"
_punctuation = "!'(),.:;? "
_special = "-"
_letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
_silences = ["@sp", "@spn", "@sil"]

# Prepend "@" to ARPAbet symbols to ensure uniqueness (some are the same as uppercase letters):
_arpabet = ["@" + s for s in cmudict.valid_symbols]
_pinyin = ["@" + s for s in pinyin.valid_symbols]

# Export all symbols:
symbols = (
    [_pad]
    + list(_special)
    + list(_punctuation)
    + list(_letters)
    + _arpabet
    + _pinyin
    + _silences
)


In [ ]:
!cat text/cleaners.py

""" from https://github.com/keithito/tacotron """

'''
Cleaners are transformations that run over the input text at both training and eval time.

Cleaners can be selected by passing a comma-delimited list of cleaner names as the "cleaners"
hyperparameter. Some cleaners are English-specific. You'll typically want to use:
  1. "english_cleaners" for English text
  2. "transliteration_cleaners" for non-English text that can be transliterated to ASCII using
     the Unidecode library (https://pypi.python.org/pypi/Unidecode)
  3. "basic_cleaners" if you do not want to transliterate (in this case, you should also update
     the symbols in symbols.py to match your data).
'''


# Regular expression matching whitespace:
import re
from unidecode import unidecode
from .numbers import normalize_numbers
_whitespace_re = re.compile(r'\s+')

# List of (regular expression, replacement) pairs for abbreviations:
_abbreviations = [(re.compile('\\b%s\\.' % x[0], re.IGNORECASE), x[1]) for x in [
    ('mrs

In [ ]:
!ls -lah

total 144K
drwxr-xr-x 15 root root 4.0K Nov  7 11:01 .
drwxr-xr-x  1 root root 4.0K Nov  7 11:01 ..
drwxr-xr-x  2 root root 4.0K Nov  7 11:01 audio
drwxr-xr-x  6 root root 4.0K Nov  7 11:01 config
-rw-r--r--  1 root root 7.8K Nov  7 11:01 dataset.py
drwxr-xr-x  5 root root 4.0K Nov  7 11:01 demo
-rw-r--r--  1 root root 3.7K Nov  7 11:01 evaluate.py
drwxr-xr-x  8 root root 4.0K Nov  7 11:01 .git
-rw-r--r--  1 root root 1.4K Nov  7 11:01 .gitignore
drwxr-xr-x  2 root root 4.0K Nov  7 11:01 hifigan
drwxr-xr-x  2 root root 4.0K Nov  7 11:01 img
-rw-r--r--  1 root root  26K Nov  7 11:01 index.html
drwxr-xr-x  2 root root 4.0K Nov  7 11:01 lexicon
-rw-r--r--  1 root root 1.1K Nov  7 11:01 LICENSE
drwxr-xr-x  2 root root 4.0K Nov  7 11:01 model
-rw-r--r--  1 root root  605 Nov  7 11:01 prepare_align.py
drwxr-xr-x  5 root root 4.0K Nov  7 11:01 preprocessed_data
drwxr-xr-x  2 root root 4.0K Nov  7 11:01 preprocessor
-rw-r--r--  1 root root  403 Nov  7 11:01 preprocess.py
-rw-r--r--  1 root roo

In [ ]:
print("=== PREPROCESSED DATA ===")
!find preprocessed_data -maxdepth 3 -type f | head -40

print("\n=== ALIGNMENT SCRIPT ===")
!cat prepare_align.py

print("\n=== README DATASET INSTRUCTIONS ===")
!grep -in -E "dataset|preprocess|align|LJSpeech|AISHELL|LibriTTS" README.md | head -50

=== PREPROCESSED DATA ===
preprocessed_data/AISHELL3/val.txt
preprocessed_data/AISHELL3/speakers.json
preprocessed_data/AISHELL3/stats.json
preprocessed_data/AISHELL3/train.txt
preprocessed_data/LibriTTS/val.txt
preprocessed_data/LibriTTS/speakers.json
preprocessed_data/LibriTTS/stats.json
preprocessed_data/LibriTTS/train.txt
preprocessed_data/LJSpeech/val.txt
preprocessed_data/LJSpeech/speakers.json
preprocessed_data/LJSpeech/stats.json
preprocessed_data/LJSpeech/train.txt

=== ALIGNMENT SCRIPT ===
import argparse

import yaml

from preprocessor import ljspeech, aishell3, libritts


def main(config):
    if "LJSpeech" in config["dataset"]:
        ljspeech.prepare_align(config)
    if "AISHELL3" in config["dataset"]:
        aishell3.prepare_align(config)
    if "LibriTTS" in config["dataset"]:
        libritts.prepare_align(config)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("config", type=str, help="path to preprocess.yaml")
    args =

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

WORK_DIR = "/content/drive/MyDrive/FastSpeech2_Assignment"

os.makedirs(WORK_DIR, exist_ok=True)

print("Persistent folder:", WORK_DIR)
print("Folder exists:", os.path.exists(WORK_DIR))

Persistent folder: /content/drive/MyDrive/FastSpeech2_Assignment
Folder exists: True


In [ ]:
print("LJSpeech folders:")
!find preprocessed_data/LJSpeech -maxdepth 2 -type d

print("\nLJSpeech training examples:")
!head -3 preprocessed_data/LJSpeech/train.txt

print("\nLJSpeech validation examples:")
!head -3 preprocessed_data/LJSpeech/val.txt

print("\nLJSpeech folder size:")
!du -sh preprocessed_data/LJSpeech

LJSpeech folders:
preprocessed_data/LJSpeech

LJSpeech training examples:
LJ030-0053|LJSpeech|{IH1 T W AH0 Z AW1 T F IH2 T IH0 D sp W IH1 TH AH0 K L IH1 R P L AE1 S T IH0 K sp spn sp W IH1 CH W AH0 Z N AY1 DH ER0 B UH1 L AH0 T P R UW2 F sp N AO1 R B UH1 L AH0 T R IH0 Z IH1 S T AH0 N T}|it was outfitted with a clear plastic bubbletop which was neither bulletproof nor bullet resistant.
LJ003-0323|LJSpeech|{D R AH1 NG K AH0 N N AH0 S sp IH1 F IH1 T EH1 V ER0 AH0 K ER1 D sp SH UH1 D B IY1 V IH1 Z IH0 T IH0 D W IH1 TH S AH0 V IH1 R P AH1 N IH0 SH M AH0 N T}|drunkenness, if it ever occurred, should be visited with severe punishment;
LJ002-0072|LJSpeech|{DH IY0 IH1 N M EY2 T S W ER0 P R IH1 V IH0 L IH0 JH D sp AY1 DH ER0 B AY1 AE2 N T IH0 S IY1 D AH0 N T S sp AO1 R DH AH1 F AO1 R CH AH0 N AH0 T P AH0 Z EH1 SH AH0 N AH0 V S AH0 F IH1 SH AH0 N T F AH1 N D Z sp T IH0 P EY1 DH AH0 CH AA1 R JH IH0 Z sp AH1 V DH AH1 P L EY1 S}|the inmates were privileged, either by antecedents or the fortunate poss

In [1]:
import os
print("Repository exists:", os.path.exists("/content/FastSpeech2"))
!nvidia-smi

Repository exists: False
Sat Nov  7 14:10:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------

In [2]:
!git clone https://github.com/ming024/FastSpeech2.git /content/FastSpeech2
%cd /content/FastSpeech2

Cloning into '/content/FastSpeech2'...
remote: Enumerating objects: 994, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 994 (delta 11), reused 7 (delta 7), pack-reused 978 (from 1)
Receiving objects: 100% (994/994), 330.31 MiB | 36.49 MiB/s, done.
Resolving deltas: 100% (176/176), done.
/content/FastSpeech2


In [3]:
import os
print("Repository restored:", os.path.exists("train.py"))

Repository restored: True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os

WORK_DIR = "/content/drive/MyDrive/FastSpeech2_Assignment"
os.makedirs(WORK_DIR, exist_ok=True)

print("Workspace ready:", WORK_DIR)

Mounted at /content/drive
Workspace ready: /content/drive/MyDrive/FastSpeech2_Assignment


In [6]:
!pip install -q datasets soundfile

In [7]:
!pip install -q -U datasets huggingface_hub soundfile

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [8]:
import datasets
import huggingface_hub
import soundfile
import torch

print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("SoundFile:", soundfile.__version__)
print("CUDA available:", torch.cuda.is_available())

Datasets: 5.1.0
Hugging Face Hub: 2.4.0
SoundFile: 0.14.0
CUDA available: True


In [9]:
from datasets import load_dataset, Audio

telugu = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

telugu = telugu.cast_column("audio", Audio(decode=False))

sample = next(iter(telugu))

print("Available fields:", sample.keys())
print("Transcript:", sample["text"])
print("Audio information:", sample["audio"].keys())
print("Gender:", sample.get("gender"))

README.md:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

Available fields: dict_keys(['audio', 'text', 'gender'])
Transcript: నరకవాసి.
Audio information: dict_keys(['bytes', 'path'])
Gender: 0


In [10]:
from itertools import islice

for i, item in enumerate(islice(iter(telugu), 5), start=1):
    print(f"\nSample {i}")
    print("Transcript:", item["text"])
    print("Gender code:", item["gender"])
    print("Audio path:", item["audio"]["path"])
    print("Audio bytes available:", item["audio"]["bytes"] is not None)


Sample 1
Transcript: నరకవాసి.
Gender code: 0
Audio path: train_telugufemale_00001.wav
Audio bytes available: True

Sample 2
Transcript: బ్రహ్మదత్తుడు కాశీరాజ్యాన్ని పరిపాలించే కాలంలో ఆ నగరంలో ధనికుడైన ఒక గొప్పవర్తకుడుండేవాడు ఆయనకు మిత్ర విందకుడని ఒక కొడుకుండే వాడు ఈ మిత్రవిందకుడు ఎంతో పాపాత్ముడు వర్తకుడు చనిపోయాక ఆయన భార్య తన కుమారుడికి నాయనా దానాలు చెయ్యి నియమాలు పాటించు ధర్మం అనుసరించు అని ఎంతో హితబోధ చేసింది కాని వాడు తల్లి మాటలు కొంచెమైనా వినిపించుకోలేదు.
Gender code: 0
Audio path: train_telugufemale_00002.wav
Audio bytes available: True

Sample 3
Transcript: ఇంతలో కార్తీక పౌర్ణమి వచ్చింది మిత్రవిందకుడితో తల్లి నాయనా ఇవాళ పుణ్యదినం తెల్లవార్లూ విహారంలో ధర్మోపదేశం చేస్తారు నీవు అక్కడ అందరితోపాటు పూజచేసుకుని ఉపదేశం వినిరా తిరిగివచ్చాక నీకు వెయ్యి రుపాయలిస్తాను అన్నది.
Gender code: 0
Audio path: train_telugufemale_00003.wav
Audio bytes available: True

Sample 4
Transcript: డబ్బుకోసమని మిత్రవిందకుడు సరేనన్నాడు అతను ఉపదేశం వినటానికి వెళ్ళాడుగాని ఒక మూల పడుకుని నిద్రపోయాడు తెల్లవారుతూనే

In [11]:
from itertools import islice
from collections import Counter
import soundfile as sf
import io

gender_counts = Counter()
durations = []
sample_rates = Counter()

for item in islice(iter(telugu), 20):
    gender_counts[item["gender"]] += 1

    audio_bytes = item["audio"]["bytes"]
    info = sf.info(io.BytesIO(audio_bytes))

    durations.append(info.duration)
    sample_rates[info.samplerate] += 1

print("Gender distribution in 20 samples:", gender_counts)
print("Sample rates:", sample_rates)
print("Average audio duration:", round(sum(durations) / len(durations), 2), "seconds")
print("Total duration inspected:", round(sum(durations), 2), "seconds")

Gender distribution in 20 samples: Counter({0: 20})
Sample rates: Counter({16000: 20})
Average audio duration: 22.89 seconds
Total duration inspected: 457.72 seconds


In [12]:
from itertools import islice
import soundfile as sf
import io

next_20_durations = []

for i, item in enumerate(islice(iter(telugu), 20, 40), start=21):
    info = sf.info(io.BytesIO(item["audio"]["bytes"]))
    next_20_durations.append(info.duration)

    print(
        f"Sample {i}: "
        f"{info.duration:.2f} seconds | "
        f"{info.samplerate} Hz | "
        f"Gender: {item['gender']}"
    )

print("\nAverage duration of samples 21–40:",
      round(sum(next_20_durations) / len(next_20_durations), 2), "seconds")

Sample 21: 26.61 seconds | 16000 Hz | Gender: 0
Sample 22: 16.00 seconds | 16000 Hz | Gender: 0
Sample 23: 13.49 seconds | 16000 Hz | Gender: 0
Sample 24: 7.06 seconds | 16000 Hz | Gender: 0
Sample 25: 39.01 seconds | 16000 Hz | Gender: 0
Sample 26: 13.91 seconds | 16000 Hz | Gender: 0
Sample 27: 2.53 seconds | 16000 Hz | Gender: 0
Sample 28: 39.66 seconds | 16000 Hz | Gender: 0
Sample 29: 19.51 seconds | 16000 Hz | Gender: 0
Sample 30: 20.06 seconds | 16000 Hz | Gender: 0
Sample 31: 6.74 seconds | 16000 Hz | Gender: 0
Sample 32: 40.78 seconds | 16000 Hz | Gender: 0
Sample 33: 41.15 seconds | 16000 Hz | Gender: 0
Sample 34: 7.08 seconds | 16000 Hz | Gender: 0
Sample 35: 38.01 seconds | 16000 Hz | Gender: 0
Sample 36: 19.32 seconds | 16000 Hz | Gender: 0
Sample 37: 28.80 seconds | 16000 Hz | Gender: 0
Sample 38: 16.35 seconds | 16000 Hz | Gender: 0
Sample 39: 53.04 seconds | 16000 Hz | Gender: 0
Sample 40: 14.10 seconds | 16000 Hz | Gender: 0

Average duration of samples 21–40: 23.16 se

In [13]:
from collections import Counter

gender_counts = Counter()

for item in telugu:
    gender_counts[item["gender"]] += 1

print("Total records:", sum(gender_counts.values()))
print("Gender distribution:", gender_counts)

Total records: 8576
Gender distribution: Counter({1: 4533, 0: 4043})


In [14]:
import os
import shutil

WORK_DIR = "/content/drive/MyDrive/FastSpeech2_Assignment"

folders = [
    "datasets/Telugu/train",
    "datasets/Telugu/val",
    "datasets/English/train",
    "datasets/English/val",
    "checkpoints/Telugu",
    "checkpoints/English",
    "results/Telugu",
    "results/English",
    "logs"
]

for folder in folders:
    os.makedirs(os.path.join(WORK_DIR, folder), exist_ok=True)

print("Workspace:", WORK_DIR)
print("Repository exists:", os.path.isdir("/content/FastSpeech2"))
print("Google Drive mounted:", os.path.isdir("/content/drive/MyDrive"))

total, used, free = shutil.disk_usage("/content")
print("Colab free storage:", round(free / (1024**3), 2), "GB")

Workspace: /content/drive/MyDrive/FastSpeech2_Assignment
Repository exists: True
Google Drive mounted: True
Colab free storage: 111.8 GB


In [15]:
import os
import io
import csv
import hashlib
import soundfile as sf
from tqdm.auto import tqdm

# Target durations
TRAIN_TARGET = 5 * 60 * 60
VAL_TARGET = 1 * 60 * 60

# Output folders
BASE = os.path.join(WORK_DIR, "datasets", "Telugu")

for split in ["train", "val"]:
    os.makedirs(os.path.join(BASE, split, "wavs"), exist_ok=True)

# Metadata files
train_csv = open(os.path.join(BASE, "train", "metadata.csv"),
                 "w", encoding="utf-8", newline="")
val_csv = open(os.path.join(BASE, "val", "metadata.csv"),
               "w", encoding="utf-8", newline="")

train_writer = csv.writer(train_csv, delimiter="|")
val_writer = csv.writer(val_csv, delimiter="|")

train_writer.writerow(["filename", "speaker", "text", "duration", "sample_rate", "gender"])
val_writer.writerow(["filename", "speaker", "text", "duration", "sample_rate", "gender"])

durations = {"train": 0.0, "val": 0.0}
counts = {"train": 0, "val": 0}
gender_counts = {"train": {}, "val": {}}

# Stream records and save only the required duration
for item in tqdm(telugu, total=8576, desc="Saving Telugu audio"):

    if durations["train"] >= TRAIN_TARGET and durations["val"] >= VAL_TARGET:
        break

    audio_bytes = item["audio"]["bytes"]

    if audio_bytes is None:
        continue

    filename = os.path.basename(item["audio"]["path"])
    gender = int(item["gender"])

    # Deterministic split: 20% validation, 80% training
    hash_value = int(hashlib.md5(filename.encode()).hexdigest(), 16)
    split = "val" if hash_value % 5 == 0 else "train"

    target = VAL_TARGET if split == "val" else TRAIN_TARGET

    if durations[split] >= target:
        continue

    info = sf.info(io.BytesIO(audio_bytes))
    duration = info.duration

    speaker = "Telugu_female" if gender == 0 else "Telugu_male"

    wav_path = os.path.join(BASE, split, "wavs", filename)

    with open(wav_path, "wb") as f:
        f.write(audio_bytes)

    writer = val_writer if split == "val" else train_writer

    writer.writerow([
        filename,
        speaker,
        item["text"].replace("\n", " ").replace("|", " "),
        round(duration, 3),
        info.samplerate,
        gender
    ])

    durations[split] += duration
    counts[split] += 1

    gender_counts[split][gender] = gender_counts[split].get(gender, 0) + 1

    if counts["train"] + counts["val"] % 100 == 0:
        train_csv.flush()
        val_csv.flush()

train_csv.close()
val_csv.close()

print("\n===== EXTRACTION SUMMARY =====")

for split in ["train", "val"]:
    print(f"\n{split.upper()}")
    print("Recordings:", counts[split])
    print("Duration (hours):", round(durations[split] / 3600, 3))
    print("Gender counts:", gender_counts[split])

print("\nSaved in:", BASE)

Saving Telugu audio:   0%|          | 0/8576 [00:00<?, ?it/s]


===== EXTRACTION SUMMARY =====

TRAIN
Recordings: 813
Duration (hours): 5.012
Gender counts: {0: 813}

VAL
Recordings: 167
Duration (hours): 1.005
Gender counts: {0: 167}

Saved in: /content/drive/MyDrive/FastSpeech2_Assignment/datasets/Telugu


In [1]:
import os
import shutil

BASE = os.path.join(WORK_DIR, "datasets", "Telugu")

for split in ["train", "val"]:
    folder = os.path.join(BASE, split)

    if os.path.exists(folder):
        shutil.rmtree(folder)

    os.makedirs(os.path.join(folder, "wavs"), exist_ok=True)

print("Old Telugu extraction cleared.")

NameError: name 'WORK_DIR' is not defined

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

WORK_DIR = "/content/drive/MyDrive/FastSpeech2_Assignment"
BASE = os.path.join(WORK_DIR, "datasets", "Telugu")

print("Workspace exists:", os.path.isdir(WORK_DIR))
print("Telugu dataset exists:", os.path.isdir(BASE))
print("Repository exists:", os.path.isdir("/content/FastSpeech2"))

Mounted at /content/drive
Workspace exists: True
Telugu dataset exists: True
Repository exists: False


In [4]:
for split in ["train", "val"]:
    folder = os.path.join(BASE, split)
    wav_folder = os.path.join(folder, "wavs")

    if os.path.isdir(wav_folder):
        files = [f for f in os.listdir(wav_folder) if f.endswith(".wav")]
        print(split, "recordings:", len(files))
    else:
        print(split, "folder missing")

train recordings: 0
val recordings: 0


In [5]:
!git clone https://github.com/ming024/FastSpeech2.git /content/FastSpeech2

!pip install -q datasets huggingface_hub soundfile librosa tqdm

from google.colab import drive
drive.mount('/content/drive')

import os
import io
import csv
import hashlib
import soundfile as sf
from datasets import load_dataset, Audio
from tqdm.auto import tqdm

WORK_DIR = "/content/drive/MyDrive/FastSpeech2_Assignment"
BASE = os.path.join(WORK_DIR, "datasets", "Telugu")

telugu = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

telugu = telugu.cast_column("audio", Audio(decode=False))

print("Repository restored:", os.path.isdir("/content/FastSpeech2"))
print("Dataset loaded:", telugu is not None)
print("Drive workspace:", WORK_DIR)

Cloning into '/content/FastSpeech2'...
remote: Enumerating objects: 994, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 994 (delta 11), reused 7 (delta 7), pack-reused 978 (from 1)
Receiving objects: 100% (994/994), 330.31 MiB | 28.23 MiB/s, done.
Resolving deltas: 100% (176/176), done.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


README.md:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

Repository restored: True
Dataset loaded: True
Drive workspace: /content/drive/MyDrive/FastSpeech2_Assignment


In [8]:
# Target 2.5 hours per gender for training
# Target 0.5 hours per gender for validation

targets = {
    "train": {0: 9000, 1: 9000},
    "val":   {0: 1800, 1: 1800}
}

os.makedirs(BASE, exist_ok=True)

for split in ["train", "val"]:
    os.makedirs(os.path.join(BASE, split, "wavs"), exist_ok=True)

writers = {}
files = {}

for split in ["train", "val"]:
    f = open(
        os.path.join(BASE, split, "metadata.csv"),
        "w", encoding="utf-8", newline=""
    )
    files[split] = f
    writers[split] = csv.writer(f, delimiter="|")
    writers[split].writerow(
        ["filename", "speaker", "text", "duration", "sample_rate", "gender"]
    )

totals = {
    "train": {0: 0.0, 1: 0.0},
    "val": {0: 0.0, 1: 0.0}
}

counts = {
    "train": {0: 0, 1: 0},
    "val": {0: 0, 1: 0}
}

for item in tqdm(telugu, total=8576, desc="Extracting balanced Telugu"):

    gender = int(item["gender"])

    if gender not in [0, 1]:
        continue

    # Decide validation/training using a stable filename hash
    filename = os.path.basename(item["audio"]["path"])
    split = (
        "val"
        if int(hashlib.md5(filename.encode()).hexdigest(), 16) % 5 == 0
        else "train"
    )

    if totals[split][gender] >= targets[split][gender]:
        continue

    audio_bytes = item["audio"]["bytes"]

    if audio_bytes is None:
        continue

    info = sf.info(io.BytesIO(audio_bytes))
    duration = info.duration

    speaker = "Telugu_female" if gender == 0 else "Telugu_male"

    wav_path = os.path.join(BASE, split, "wavs", filename)

    with open(wav_path, "wb") as wav:
        wav.write(audio_bytes)

    writers[split].writerow([
        filename,
        speaker,
        item["text"].replace("\n", " ").replace("|", " "),
        round(duration, 3),
        info.samplerate,
        gender
    ])

    totals[split][gender] += duration
    counts[split][gender] += 1

    for f in files.values():
        f.flush()

    if all(
        totals[s][g] >= targets[s][g]
        for s in ["train", "val"]
        for g in [0, 1]
    ):
        break

for f in files.values():
    f.close()

print("\n===== FINAL TELUGU DATASET =====")

for split in ["train", "val"]:
    for gender in [0, 1]:
        print(
            split,
            "Female" if gender == 0 else "Male",
            "hours:", round(totals[split][gender] / 3600, 3),
            "recordings:", counts[split][gender]
        )

Extracting balanced Telugu:   0%|          | 0/8576 [00:00<?, ?it/s]


===== FINAL TELUGU DATASET =====
train Female hours: 2.503 recordings: 419
train Male hours: 2.502 recordings: 1106
val Female hours: 0.513 recordings: 81
val Male hours: 0.501 recordings: 240


In [9]:
import os
import csv

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment/datasets/Telugu"

for split in ["train", "val"]:
    wav_dir = os.path.join(BASE, split, "wavs")
    metadata = os.path.join(BASE, split, "metadata.csv")

    wav_count = len([
        f for f in os.listdir(wav_dir)
        if f.endswith(".wav")
    ])

    with open(metadata, encoding="utf-8") as f:
        metadata_count = sum(1 for _ in f) - 1

    print(f"{split}:")
    print("  WAV files:", wav_count)
    print("  Metadata entries:", metadata_count)

train:
  WAV files: 1525
  Metadata entries: 1525
val:
  WAV files: 321
  Metadata entries: 321


In [10]:
from datasets import load_dataset, Audio

svarah = load_dataset(
    "ai4bharat/Svarah",
    split="test",
    streaming=True
)

svarah = svarah.cast_column(
    "audio_filepath",
    Audio(decode=False)
)

print("Dataset loaded successfully!")
print("Features:", svarah.features)

sample = next(iter(svarah))

print("\nSample transcript:")
print(sample["text"])

print("\nAudio information:")
print("Audio keys:", sample["audio_filepath"].keys())
print("Audio path:", sample["audio_filepath"]["path"])
print("Audio bytes available:", sample["audio_filepath"]["bytes"] is not None)

print("\nOther metadata:")
print("Duration:", sample["duration"])
print("Gender:", sample.get("gender"))

README.md:   0%|          | 0.00/4.30k [00:00<?, ?B/s]

DatasetNotFoundError: Dataset 'ai4bharat/Svarah' is a gated dataset on the Hub. You must be authenticated to access it.

In [11]:
from huggingface_hub import notebook_login

notebook_login()

In [12]:
from datasets import load_dataset, Audio

svarah = load_dataset(
    "ai4bharat/Svarah",
    split="test",
    streaming=True,
    token=True
)

svarah = svarah.cast_column(
    "audio_filepath",
    Audio(decode=False)
)

sample = next(iter(svarah))

print("Dataset loaded successfully!")
print("Transcript:", sample["text"])
print("Audio available:", sample["audio_filepath"]["bytes"] is not None)
print("Duration:", sample["duration"])
print("Gender:", sample["gender"])

Dataset loaded successfully!
Transcript: Our district Hasana was the land of the great Hoysala dynasty who ruled over this region from 11th to 14th century CE.
Audio available: True
Duration: 8.7595
Gender: Female


In [13]:
import os
import io
import csv
import hashlib
import soundfile as sf
from tqdm.auto import tqdm

WORK_DIR = "/content/drive/MyDrive/FastSpeech2_Assignment"
BASE_EN = os.path.join(WORK_DIR, "datasets", "English")

# Assignment targets
targets = {
    "train": 5 * 3600,
    "val": 1 * 3600
}

os.makedirs(os.path.join(BASE_EN, "train", "wavs"), exist_ok=True)
os.makedirs(os.path.join(BASE_EN, "val", "wavs"), exist_ok=True)

writers = {}
files = {}

for split in ["train", "val"]:
    f = open(
        os.path.join(BASE_EN, split, "metadata.csv"),
        "w", encoding="utf-8", newline=""
    )
    files[split] = f
    writers[split] = csv.writer(f, delimiter="|")
    writers[split].writerow(
        ["filename", "speaker", "text", "duration", "sample_rate", "gender"]
    )

totals = {"train": 0.0, "val": 0.0}
counts = {"train": 0, "val": 0}

for index, item in enumerate(
    tqdm(svarah, desc="Extracting English Svarah")
):

    # Stop when both targets are reached
    if all(totals[s] >= targets[s] for s in targets):
        break

    audio = item["audio_filepath"]
    audio_bytes = audio["bytes"]

    if audio_bytes is None:
        continue

    info = sf.info(io.BytesIO(audio_bytes))
    duration = info.duration

    # Stable 5:1 split between training and validation
    source_name = audio.get("path") or f"sample_{index}"
    split_hash = int(
        hashlib.md5(source_name.encode()).hexdigest(), 16
    )
    split = "val" if split_hash % 6 == 0 else "train"

    if totals[split] >= targets[split]:
        continue

    gender = str(item.get("gender", "Unknown"))
    speaker = "English_" + gender.capitalize()

    filename = f"svarah_{index:06d}.wav"
    wav_path = os.path.join(BASE_EN, split, "wavs", filename)

    with open(wav_path, "wb") as wav:
        wav.write(audio_bytes)

    text = str(item["text"]).replace("\n", " ").replace("|", " ")

    writers[split].writerow([
        filename,
        speaker,
        text,
        round(duration, 3),
        info.samplerate,
        gender
    ])

    totals[split] += duration
    counts[split] += 1

    for f in files.values():
        f.flush()

for f in files.values():
    f.close()

print("\n===== FINAL ENGLISH DATASET =====")

for split in ["train", "val"]:
    print(
        split,
        "hours:", round(totals[split] / 3600, 3),
        "recordings:", counts[split]
    )

Extracting English Svarah: 0it [00:00, ?it/s]


===== FINAL ENGLISH DATASET =====
train hours: 5.001 recordings: 3481
val hours: 1.002 recordings: 713


In [14]:
import os

BASE_EN = "/content/drive/MyDrive/FastSpeech2_Assignment/datasets/English"

for split in ["train", "val"]:
    wav_dir = os.path.join(BASE_EN, split, "wavs")
    metadata = os.path.join(BASE_EN, split, "metadata.csv")

    wav_count = len([
        f for f in os.listdir(wav_dir)
        if f.endswith(".wav")
    ])

    with open(metadata, encoding="utf-8") as f:
        metadata_count = sum(1 for _ in f) - 1

    print(f"{split}:")
    print("  WAV files:", wav_count)
    print("  Metadata entries:", metadata_count)

train:
  WAV files: 3481
  Metadata entries: 3482
val:
  WAV files: 713
  Metadata entries: 713


In [15]:
import os
import csv
from collections import Counter

BASE_EN = "/content/drive/MyDrive/FastSpeech2_Assignment/datasets/English"

for split in ["train", "val"]:
    wav_dir = os.path.join(BASE_EN, split, "wavs")
    metadata = os.path.join(BASE_EN, split, "metadata.csv")

    with open(metadata, encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f, delimiter="|"))

    filenames = [row["filename"] for row in rows]
    duplicates = [
        name for name, count in Counter(filenames).items()
        if count > 1
    ]

    missing = [
        name for name in filenames
        if not os.path.isfile(os.path.join(wav_dir, name))
    ]

    print(f"\n{split.upper()}")
    print("CSV records:", len(rows))
    print("WAV files:", len([
        name for name in os.listdir(wav_dir)
        if name.endswith(".wav")
    ]))
    print("Duplicate filenames:", duplicates[:10])
    print("Missing WAV files:", missing[:10])


TRAIN
CSV records: 3481
WAV files: 3481
Duplicate filenames: []
Missing WAV files: []

VAL
CSV records: 713
WAV files: 713
Duplicate filenames: []
Missing WAV files: []


In [16]:
import os
import csv
import soundfile as sf
from collections import Counter

WORK_DIR = "/content/drive/MyDrive/FastSpeech2_Assignment"

for language in ["English", "Telugu"]:
    print(f"\n===== {language.upper()} DATASET =====")

    for split in ["train", "val"]:
        base = os.path.join(WORK_DIR, "datasets", language, split)
        wav_dir = os.path.join(base, "wavs")
        metadata = os.path.join(base, "metadata.csv")

        sample_rates = Counter()
        empty_text = 0
        total_duration = 0

        with open(metadata, encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f, delimiter="|"))

        for row in rows:
            text = row["text"].strip()

            if not text:
                empty_text += 1

            path = os.path.join(wav_dir, row["filename"])
            info = sf.info(path)

            sample_rates[info.samplerate] += 1
            total_duration += info.duration

        print(f"\n{split.upper()}")
        print("Recordings:", len(rows))
        print("Sample rates:", dict(sample_rates))
        print("Empty transcripts:", empty_text)
        print("Actual audio hours:", round(total_duration / 3600, 3))


===== ENGLISH DATASET =====

TRAIN
Recordings: 3481
Sample rates: {16000: 3481}
Empty transcripts: 0
Actual audio hours: 5.001

VAL
Recordings: 713
Sample rates: {16000: 713}
Empty transcripts: 0
Actual audio hours: 1.002

===== TELUGU DATASET =====

TRAIN
Recordings: 1525
Sample rates: {16000: 419, 48000: 1106}
Empty transcripts: 0
Actual audio hours: 5.005

VAL
Recordings: 321
Sample rates: {16000: 81, 48000: 240}
Empty transcripts: 0
Actual audio hours: 1.014


In [17]:
import os
import csv
import soundfile as sf
import numpy as np
from scipy.signal import resample_poly
from math import gcd
from tqdm.auto import tqdm

WORK_DIR = "/content/drive/MyDrive/FastSpeech2_Assignment"
TARGET_SR = 22050

for language in ["English", "Telugu"]:

    print(f"\n===== PROCESSING {language.upper()} =====")

    source_base = os.path.join(WORK_DIR, "datasets", language)
    output_base = os.path.join(WORK_DIR, "datasets", language + "_22050")

    for split in ["train", "val"]:

        source_wavs = os.path.join(source_base, split, "wavs")
        output_wavs = os.path.join(output_base, split, "wavs")

        os.makedirs(output_wavs, exist_ok=True)

        source_csv = os.path.join(source_base, split, "metadata.csv")
        output_csv = os.path.join(output_base, split, "metadata.csv")

        with open(source_csv, encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f, delimiter="|"))

        with open(output_csv, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f, delimiter="|")
            writer.writerow([
                "filename", "speaker", "text",
                "duration", "sample_rate", "gender"
            ])

            for row in tqdm(rows, desc=f"{language} {split}"):

                filename = row["filename"]
                input_path = os.path.join(source_wavs, filename)
                output_path = os.path.join(output_wavs, filename)

                audio, sr = sf.read(input_path, dtype="float32")

                # Convert stereo to mono if needed
                if audio.ndim == 2:
                    audio = np.mean(audio, axis=1)

                # Resample to 22050 Hz
                if sr != TARGET_SR:
                    common = gcd(sr, TARGET_SR)
                    audio = resample_poly(
                        audio,
                        TARGET_SR // common,
                        sr // common
                    )

                sf.write(
                    output_path,
                    audio,
                    TARGET_SR,
                    subtype="PCM_16"
                )

                writer.writerow([
                    filename,
                    row["speaker"],
                    row["text"],
                    round(len(audio) / TARGET_SR, 3),
                    TARGET_SR,
                    row["gender"]
                ])

    print(f"Completed: {language}")


===== PROCESSING ENGLISH =====


English train:   0%|          | 0/3481 [00:00<?, ?it/s]

English val:   0%|          | 0/713 [00:00<?, ?it/s]

Completed: English

===== PROCESSING TELUGU =====


Telugu train:   0%|          | 0/1525 [00:00<?, ?it/s]

Telugu val:   0%|          | 0/321 [00:00<?, ?it/s]

Completed: Telugu


In [18]:
import os
import soundfile as sf
from collections import Counter

WORK_DIR = "/content/drive/MyDrive/FastSpeech2_Assignment"

for language in ["English_22050", "Telugu_22050"]:
    print(f"\n===== {language} =====")

    for split in ["train", "val"]:
        wav_dir = os.path.join(
            WORK_DIR, "datasets", language, split, "wavs"
        )

        rates = Counter()
        files = [
            f for f in os.listdir(wav_dir)
            if f.endswith(".wav")
        ]

        for filename in files:
            info = sf.info(os.path.join(wav_dir, filename))
            rates[info.samplerate] += 1

        print(split, "files:", len(files))
        print("Sample rates:", dict(rates))


===== English_22050 =====
train files: 3481
Sample rates: {22050: 3481}
val files: 713
Sample rates: {22050: 713}

===== Telugu_22050 =====
train files: 1525
Sample rates: {22050: 1525}
val files: 321
Sample rates: {22050: 321}


In [19]:
import os

REPO = "/content/FastSpeech2"

files_to_inspect = [
    "preprocess.py",
    "preprocessor/preprocessor.py",
    "preprocessor/ljspeech.py",
    "config/LJSpeech/preprocess.yaml",
    "text/symbols.py",
]

for filename in files_to_inspect:
    path = os.path.join(REPO, filename)
    print("\n" + "=" * 80)
    print("FILE:", filename)
    print("=" * 80)

    if not os.path.exists(path):
        print("FILE NOT FOUND")
        continue

    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # Print up to 180 lines per file
    print("".join(lines[:180]))

    if len(lines) > 180:
        print(f"\n... {len(lines) - 180} more lines not shown ...")


FILE: preprocess.py
import argparse

import yaml

from preprocessor.preprocessor import Preprocessor


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("config", type=str, help="path to preprocess.yaml")
    args = parser.parse_args()

    config = yaml.load(open(args.config, "r"), Loader=yaml.FullLoader)
    preprocessor = Preprocessor(config)
    preprocessor.build_from_path()


FILE: preprocessor/preprocessor.py
import os
import random
import json

import tgt
import librosa
import numpy as np
import pyworld as pw
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import audio as Audio


class Preprocessor:
    def __init__(self, config):
        self.config = config
        self.in_dir = config["path"]["raw_path"]
        self.out_dir = config["path"]["preprocessed_path"]
        self.val_size = config["preprocessing"]["val_size"]
        self.sampling_rate = config["preprocessing"]

In [20]:
files_to_inspect = [
    "preprocessor/preprocessor.py",
    "prepare_align.py",
    "text/__init__.py",
    "text/cleaners.py",
]

for filename in files_to_inspect:
    path = os.path.join(REPO, filename)

    print("\n" + "=" * 70)
    print("FILE:", filename)
    print("=" * 70)

    if not os.path.exists(path):
        print("Not found")
        continue

    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    if filename.endswith("preprocessor.py"):
        start, end = 175, len(lines)
    else:
        start, end = 0, len(lines)

    print("".join(lines[start:end]))


FILE: preprocessor/preprocessor.py

        # Read raw text
        with open(text_path, "r") as f:
            raw_text = f.readline().strip("\n")

        # Compute fundamental frequency
        pitch, t = pw.dio(
            wav.astype(np.float64),
            self.sampling_rate,
            frame_period=self.hop_length / self.sampling_rate * 1000,
        )
        pitch = pw.stonemask(wav.astype(np.float64), pitch, t, self.sampling_rate)

        pitch = pitch[: sum(duration)]
        if np.sum(pitch != 0) <= 1:
            return None

        # Compute mel-scale spectrogram and energy
        mel_spectrogram, energy = Audio.tools.get_mel_from_wav(wav, self.STFT)
        mel_spectrogram = mel_spectrogram[:, : sum(duration)]
        energy = energy[: sum(duration)]

        if self.pitch_phoneme_averaging:
            # perform linear interpolation
            nonzero_ids = np.where(pitch != 0)[0]
            interp_fn = interp1d(
                nonzero_ids,
                pitc

In [21]:
import shutil
import subprocess

mfa_path = shutil.which("mfa")

if mfa_path:
    print("MFA FOUND:", mfa_path)
    subprocess.run(["mfa", "version"])
else:
    print("MFA NOT INSTALLED")

print("\nChecking alignment-related packages:")

for package in ["tgt", "textgrid", "pyworld"]:
    try:
        module = __import__(package)
        print(package, "AVAILABLE")
    except ImportError:
        print(package, "NOT INSTALLED")

MFA NOT INSTALLED

Checking alignment-related packages:
tgt NOT INSTALLED
textgrid NOT INSTALLED
pyworld NOT INSTALLED
